# 1-1 파도 — 예측하고 확인하기

규칙: **각 실험 셀을 돌리기 전에 결과를 먼저 적어 보세요.** 예측이 틀린 곳이 배우는 곳입니다.

구현: `src/rescue_sim/rescue_sim/ocean/spectrum.py`, `waves.py`  
아래 `autoreload` 덕분에 `.py` 를 고치고 저장하면 셀을 다시 돌리는 것만으로 반영됩니다.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib import font_manager
_kfont = Path("/mnt/c/Windows/Fonts/malgun.ttf")   # WSL: 윈도우 맑은 고딕으로 한글 표시
if _kfont.exists():
    font_manager.fontManager.addfont(str(_kfont))
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
from rescue_sim import CONFIG_DIR
from rescue_sim.ocean import OceanConfig, WaveField
from rescue_sim.ocean.spectrum import jonswap, sample_components, dispersion_k

harbor = OceanConfig.from_yaml(CONFIG_DIR / "ocean.yaml", preset="harbor").waves
rough  = OceanConfig.from_yaml(CONFIG_DIR / "ocean.yaml", preset="rough").waves
harbor

## 실험 1. 스펙트럼 모양 — γ 를 바꾸면?

γ(gamma)는 "봉우리 뾰족함"입니다. Hs 는 그대로 두고 γ = 1, 3.3, 7 을 비교합니다.

**예측:** γ=7 일 때 봉우리 높이는? 곡선 아래 면적(=에너지)은 γ 따라 변할까?

In [ ]:
wp = 2*np.pi/harbor.tp
w = np.linspace(0.3*wp, 3.5*wp, 500)
for g in [1.0, 3.3, 7.0]:
    S = jonswap(w, harbor.hs, harbor.tp, g)
    plt.plot(w/wp, S, label=f"γ={g}  →  Hs={4*np.sqrt(np.trapezoid(S, w)):.3f} m")
plt.xlabel("ω / ωp"); plt.ylabel("S(ω) [m²·s]"); plt.legend(); plt.title("JONSWAP (Hs=0.2, Tp=3)");

## 실험 2. 성분파 32개는 어떻게 생겼나

스펙트럼을 막대 32개로 쪼갠 게 성분파입니다. 막대 높이 = 진폭 a_i.

In [ ]:
c = sample_components(harbor)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].stem(c.omega/wp, c.amp); ax[0].set_xlabel("ω / ωp"); ax[0].set_ylabel("amp [m]")
ax[0].set_title(f"성분파 {c.n}개, 재계산 Hs = {c.hs():.3f} m")
ax[1].hist(np.rad2deg(c.theta), bins=12); ax[1].set_xlabel("진행 방향 [deg]"); ax[1].set_title("방향 (180° = 해안 쪽)");

## 실험 3. 한 점에서 본 수면 η(t)

부표 하나가 x=3, y=-2 에 떠 있다고 생각하세요. 60초 동안 위아래로 얼마나 움직일까?

**예측:** harbor 에서 가장 높은 마루는 몇 cm? rough 는? (힌트: Hs 는 '상위 1/3 파고 평균'이고, 파고 = 마루 + 골)

In [ ]:
t = np.arange(0, 60, 0.05)
for name, cfg in [("harbor", harbor), ("rough", rough)]:
    wf = WaveField(cfg, depth=5.0)
    eta = np.array([wf.elevation(3.0, -2.0, ti) for ti in t])
    plt.plot(t, eta, label=f"{name}: max {eta.max():.2f} m")
plt.axhline(0.5, ls="--", c="r", lw=0.8, label="수면 hfield 표현 한계 (+0.5 m)")
plt.axhline(-0.5, ls="--", c="r", lw=0.8)
plt.xlabel("t [s]"); plt.ylabel("η [m]"); plt.legend();

빨간 점선이 MuJoCo 수면 hfield 가 그릴 수 있는 범위입니다 (`assets/ocean_scene.xml`). 이게 1-2 에서 고칠 첫 번째 문제입니다.

## 실험 4. 50×50 m 바다 위에서 본 순간 모습

In [ ]:
X, Y = np.meshgrid(np.linspace(-25, 25, 128), np.linspace(-25, 25, 128))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, (name, cfg) in zip(ax, [("harbor", harbor), ("rough", rough)]):
    eta = WaveField(cfg, depth=5.0).elevation(X, Y, 0.0)
    im = a.imshow(eta, extent=[-25, 25, -25, 25], origin="lower", cmap="RdBu_r")
    a.set_title(name); a.set_xlabel("x [m] (← 해안 | 먼바다 →)"); a.set_ylabel("y [m]")
    fig.colorbar(im, ax=a, label="η [m]")

## 실험 5. 망가뜨려 보기 ← 직접 바꿔 보세요

아래 값을 하나씩 바꿔서 실험 4 모습이 어떻게 변하는지 보세요. 돌리기 전에 먼저 예측하세요.

| 바꿀 것 | 예측해 볼 질문 |
|---|---|
| `n_components=4` | 바다처럼 보일까, 무늬처럼 보일까? |
| `spreading_deg=1` | 마루선 모양은? |
| `phase` 를 전부 0 | t=0 에서 원점 근처에 무슨 일이? |

In [ ]:
from dataclasses import replace
cfg = replace(harbor, n_components=4)          # ← 여기를 바꿔 보세요
c = sample_components(cfg)
# c.phase[:] = 0.0                              # ← 위상 실험은 이 줄 주석 해제
eta = WaveField(cfg, components=c).elevation(X, Y, 0.0)
plt.imshow(eta, extent=[-25, 25, -25, 25], origin="lower", cmap="RdBu_r"); plt.colorbar(label="η [m]");

## 실험 6. MuJoCo 관점: 이 계산은 얼마나 비싼가

수면을 128×128 격자로 25fps 갱신한다고 합시다. MuJoCo 스텝만 돌리면 실시간보다 약 2000배 빠릅니다.

**예측:** 파도 계산을 붙이면 몇 배속까지 떨어질까? (2000? 200? 20?)

In [ ]:
import time
wf = WaveField(harbor)
for res in [32, 64, 128, 256]:
    Xr, Yr = np.meshgrid(np.linspace(-25, 25, res), np.linspace(-25, 25, res))
    t0 = time.perf_counter()
    for i in range(20): wf.elevation(Xr, Yr, i*0.04)
    ms = (time.perf_counter() - t0) / 20 * 1e3
    cost = ms * 25 / 1e3   # sim 1초당 파도 계산에 드는 실제 초
    print(f"{res:>3}²: {ms:6.2f} ms/frame → 최대 약 {1/cost:6.1f}x realtime (파도 계산만)")

## 정리 — 본인 말로 세 줄

1. 바다 = ...
2. Hs, Tp, γ 는 각각 ...
3. MuJoCo 에 붙일 때 문제가 될 것 같은 점은 ...
